This notebook translates the filtered Italian parliamentary speeches to English and produces hierarchical summaries grouped by topic, source, party family, and year. The inputs are the top-speech files produced in the data preparation notebook.


We started the generation task from just the sentences, as a proxy for the extraction layer of the summarization task. This approach seemed to us optimal as it would greatly shorten the computations and it would also ensure that the content used for the summarization was relevant to the chosen topics.

However, the output of the task was non-sensical due to the lack of proper context so we had to restart from the speeches.
We previously selected the top 100 speeches for each group (topix x source x year x party_family) in order of relevance to the topic. We used these speeches, translated them to English, and proceeded to perform hierarchical summarization.

Hierarchical summarization has the known issue of losing the logical ties between sentences. So the output is a summary table made up of coherent sentences which are not connected with each other.

# Create the Environment and functions for later


In [ ]:
# Run once. Then run again from CELL 2.
!pip install transformers sentencepiece sacremoses torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 50.7 MB/s eta 0:00:00


In [ ]:
#IMPORTS AND DEVICE
import os
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import MarianMTModel, MarianTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")
print("Files in /content:")
print(os.listdir('/content')[:30])

Running on: cuda
Files in /content:
['.config', 'df_for_summarization (1).pkl', 'sample_data']


# Translation

In order to perform summarization we need to have all the data in the same language, hence we are going to translate the parliamentary speeches in English using *"Helsinki-NLP/opus-mt-it-en"*.

English is preferred compared to Italian as it expands our choice of models for the generation task.

In [ ]:
# LOAD TRANSLATION MODEL: Italian to English
MODEL_NAME = "Helsinki-NLP/opus-mt-it-en"

translator_tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
translator_model = MarianMTModel.from_pretrained(MODEL_NAME).to(device)
translator_model.eval()

print("Translation model loaded:", MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/814k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/344M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Translation model loaded: Helsinki-NLP/opus-mt-it-en


In [ ]:
# define the TRANSLATION FUNCTION
def translate_it_en(texts, batch_size=32):
    """Translate a list of Italian texts to English."""
    translations = []
    texts = list(texts)

    for i in tqdm   (range(0, len(texts), batch_size), desc="Translating IT to EN"):
        batch = texts[i:i + batch_size]
        batch = [str(t) if pd.notna(t) and str(t).strip() else "[EMPTY]" for t in batch]

        inputs = translator_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            translated = translator_model.generate(
                **inputs,
                max_length=512,
                num_beams=4,
                early_stopping=True
            )

        decoded = translator_tokenizer.batch_decode(translated, skip_special_tokens=True)
        translations.extend(decoded)

    return translations

import the files to translate.

In [ ]:
from google.colab import files
import os
import pandas as pd

df_for_translation_m = pd.read_pickle('/content/top_speeches_m.pkl')
df_for_translation_w = pd.read_pickle('/content/top_speeches_w.pkl')

print("Migration dataframe:", df_for_translation_m.shape)
print("War dataframe:", df_for_translation_w.shape)

Migration dataframe: (9766, 11)
War dataframe: (10033, 11)


In [ ]:
#sentences translation
'''
# ------------------------------------------------------------
# 2. Make sure there is a text column to translate
# ------------------------------------------------------------

if "sentence" not in df_for_translation_m.columns:
        raise ValueError("Migration dataframe has no text_clean or text column.")

if "sentence" not in df_for_translation_w.columns:

        raise ValueError("War dataframe has no text_clean or text column.")

# ------------------------------------------------------------
# 3. Translate only if text_en does not already exist
# ------------------------------------------------------------

if "sent_en" not in df_for_translation_m.columns:
    print("Translating migration parliament speeches...")
    df_for_translation_m["sent_en"] = translate_it_en(
        df_for_translation_m["sentence"].astype(str).tolist()
    )
else:
    print("Migration already has sent_en. Skipping translation.")

if "sent_en" not in df_for_translation_w.columns:
    print("Translating war parliament speeches...")
    df_for_translation_w["sent_en"] = translate_it_en(
        df_for_translation_w["sentence"].astype(str).tolist()
    )
else:
    print("War already has sent_en. Skipping translation.")

# ------------------------------------------------------------
# 4. Save translated datasets
# ------------------------------------------------------------

df_for_translation_m.to_pickle("/content/df_final_m_translated_sent.pkl")
df_for_translation_w.to_pickle("/content/df_final_w_translated_sent.pkl")

df_for_translation_m.to_csv(
    "/content/df_final_m_translated_sent.csv",
    index=False,
    encoding="utf-8-sig"
)

df_for_translation_w.to_csv(
    "/content/df_final_w_translated_sent.csv",
    index=False,
    encoding="utf-8-sig"
)



# ------------------------------------------------------------
# 5. Download translated datasets to your computer
# ------------------------------------------------------------

files.download("/content/df_final_m_translated_sent.pkl")
files.download("/content/df_final_w_translated_sent.pkl")

# Optional CSV downloads for Excel inspection
files.download("/content/df_final_m_translated_sent.csv")
files.download("/content/df_final_w_translated_sent.csv")
'''

Translating migration parliament speeches...


Translating IT to EN:   0%|          | 0/306 [00:00<?, ?it/s]

Translating war parliament speeches...


Translating IT to EN:   0%|          | 0/314 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# ------------------------------------------------------------
# verify there is a text column to translate
# ------------------------------------------------------------

if "text_clean" not in df_for_translation_m.columns:
    if "text" in df_for_translation_m.columns:
        df_for_translation_m["text_clean"] = df_for_translation_m["text"]
    else:
        raise ValueError("Migration dataframe has no text_clean or text column.")

if "text_clean" not in df_for_translation_w.columns:
    if "text" in df_for_translation_w.columns:
        df_for_translation_w["text_clean"] = df_for_translation_w["text"]
    else:
        raise ValueError("War dataframe has no text_clean or text column.")

# ------------------------------------------------------------
# Translate only if text_en does not already exist
# ------------------------------------------------------------

if "text_en" not in df_for_translation_m.columns:
    print("Translating migration parliament speeches...")
    df_for_translation_m["text_en"] = translate_it_en(
        df_for_translation_m["text_clean"].astype(str).tolist()
    )
else:
    print("Migration already has text_en. Skipping translation.")

if "text_en" not in df_for_translation_w.columns:
    print("Translating war parliament speeches...")
    df_for_translation_w["text_en"] = translate_it_en(
        df_for_translation_w["text_clean"].astype(str).tolist()
    )
else:
    print("War already has text_en. Skipping translation.")

# ------------------------------------------------------------
# Save translated datasets
# ------------------------------------------------------------

df_for_translation_m.to_pickle("/content/df_final_m_translated_speech.pkl")
df_for_translation_w.to_pickle("/content/df_final_w_translated_speech.pkl")

df_for_translation_m.to_csv(
    "/content/df_final_m_translated_speech.csv",
    index=False,
    encoding="utf-8-sig"
)

df_for_translation_w.to_csv(
    "/content/df_final_w_translated_speech.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved files:")
for path in [
    "/content/df_final_m_translated_speech.pkl",
    "/content/df_final_w_translated_speech.pkl",
    "/content/df_final_m_translated_speech.csv",
    "/content/df_final_w_translated_speech.csv",
]:
    print(path, round(os.path.getsize(path) / 1_000_000, 2), "MB")



# Summarization

The summarization follows a hierarchical reduction strategy to handle the token limit of BART (1,024 tokens):
1. Each speech is split into overlapping chunks of ≤900 tokens.
2. Each chunk is summarized independently.
3. Chunk summaries are merged and recursively reduced until a single summary per speech is obtained.
4. All speech summaries within a group (topic × source × party family × year) are further reduced to one final group-level summary.


Load parliament datasets -> to avoid doing everything from the beginning, if you ran the steps above ignore this cell

In [ ]:

# These are your filtered sentence-level datasets.
df_m = pd.read_pickle('/content/df_final_m_translated_speech.pkl')   # migration
df_w = pd.read_pickle('/content/df_final_w_translated_speech.pkl')   # war/defence

print(f"Migration rows: {len(df_m):,}")
print(f"War/defence rows: {len(df_w):,}")
print("Migration columns:", df_m.columns.tolist())
print("War/defence columns:", df_w.columns.tolist())

Migration rows: 9,766
War/defence rows: 10,033
Migration columns: ['row_id', 'speaker', 'party_family', 'text', 'year', 'text_clean', 'speech_id', 'sentence', 'sentence_id', 'keyword_hits', 'n_keyword_hits', 'text_en']
War/defence columns: ['row_id', 'speaker', 'party_family', 'text', 'year', 'text_clean', 'speech_id', 'sentence', 'sentence_id', 'keyword_hits', 'n_keyword_hits', 'text_en']


If data is too granular by using party family you can map it to a party group, in our case we decided to stick with party family

In [ ]:
'''# CELL 6 - MAP PARLIAMENT PARTY FAMILIES TO BROADER GROUPS
def map_party_group(x):
    if x == "PRESIDENTE":
        return "presidente"
    elif x in ["Communist/Far-left", "Social democracy", "Liberal"]:
        return "left"
    elif x in ["Christian democracy", "Conservative", "Right-wing"]:
        return "right"
    else:
        return "other"

df_m = df_m.copy()
df_w = df_w.copy()
df_m["party_group"] = df_m["party_family"].apply(map_party_group)
df_w["party_group"] = df_w["party_family"].apply(map_party_group)

print("Migration party groups:")
print(df_m["party_group"].value_counts(dropna=False))
print("\nWar/defence party groups:")
print(df_w["party_group"].value_counts(dropna=False))
'''

Migration party groups:
party_group
right         3279
left          1578
presidente     652
other          304
Name: count, dtype: int64

War/defence party groups:
party_group
right         2583
presidente    2096
left          1878
other          315
Name: count, dtype: int64


## Load and fix UN datasets

In [ ]:
# CELL 10 - LOAD UN DATASETS
# These are your filtered UN sentence-level datasets.
un_df = pd.read_pickle('/content/df_f1_m_un.pkl')
un_2_df = pd.read_pickle('/content/df_f1_w_un.pkl')

print(f"UN topic 1 rows: {len(un_df):,}")
print(f"UN topic 2 rows: {len(un_2_df):,}")
print("UN topic 1 columns:", un_df.columns.tolist())
print("UN topic 2 columns:", un_2_df.columns.tolist())

UN topic 1 rows: 64
UN topic 2 rows: 156
UN topic 1 columns: ['row_id', 'DATE', 'TEXT', 'text_clean', 'sentence', 'sentence_id', 'keyword_hits', 'n_keyword_hits']
UN topic 2 columns: ['row_id', 'DATE', 'TEXT', 'text_clean', 'sentence', 'sentence_id', 'keyword_hits', 'n_keyword_hits']


In [ ]:
un_topic1_df=un_df.copy()
un_topic2_df=un_2_df.copy()

In [ ]:
# CRITICAL FIX FOR UN METADATA
# UN has no party family. If party_family stays NaN, pandas groupby drops UN later.
un_topic1_df["source"] = "UN"
un_topic2_df["source"] = "UN"

un_topic1_df["topic"] = 1
un_topic2_df["topic"] = 2

un_topic1_df["party_family"] = "ALL"
un_topic2_df["party_family"] = "ALL"

print(un_topic1_df[["row_id", "DATE", "topic", "party_family", "source"]].head())
print(un_topic2_df[["row_id", "DATE", "topic", "party_family", "source"]].head())

   row_id  DATE  topic party_family source
0       0  2006      1          ALL     UN
1       0  2006      1          ALL     UN
2       5  2011      1          ALL     UN
3       6  2012      1          ALL     UN
4       6  2012      1          ALL     UN
   row_id  DATE  topic party_family source
0       0  2006      2          ALL     UN
1       0  2006      2          ALL     UN
2       0  2006      2          ALL     UN
3       0  2006      2          ALL     UN
4       0  2006      2          ALL     UN


## Combine parliament and UN, necessary for summarization

In [ ]:

# Parliament metadata
df_for_translation_m["source"] = "parliament"
df_for_translation_w["source"] = "parliament"
df_for_translation_m["topic"] = 1
df_for_translation_w["topic"] = 2
df_for_translation_m["party_family"] = df_for_translation_m["party_family"].fillna("other")
df_for_translation_w["party_family"] = df_for_translation_w["party_family"].fillna("other")

# Combine parliament and UN
df = pd.concat(
    [df_for_translation_m, df_for_translation_w, un_topic1_df, un_topic2_df],
    ignore_index=True,
    sort=False
)

# Safety fixes
df["source"] = df["source"].fillna("unknown")
df["topic"] = df["topic"].fillna("unknown")

df["date_merged"] = ( #merge the column that contains the speech in parliament (text_en since it was translated), and text_clean for UN
    df["year"]
    .combine_first(df["DATE"])
)

# One safe group key. This is what we will group by for summarization.
df["group_key"] = (
    "topic" + df["topic"].astype(str)
    + "__" + df["party_family"].astype(str)
    + "__" + df["source"].astype(str)
    + "__" + df["date_merged"].astype(str)
)

print("Rows after concat:", len(df))
print("\nRows by source:")
print(df["source"].value_counts(dropna=False))
print("\nRows by group_key:")
print(df["group_key"].value_counts(dropna=False))

Rows after concat: 20019

Rows by source:
source
parliament    19799
UN              220
Name: count, dtype: int64

Rows by group_key:
group_key
topic2__M5S__parliament__2019.0                    100
topic2__Right-wing__parliament__2019.0             100
topic2__Social democracy__parliament__2019.0       100
topic2__Conservative__parliament__2020.0           100
topic2__Conservative__parliament__2018.0           100
                                                  ... 
topic1__ALL__UN__2006.0                              2
topic2__Christian democracy__parliament__2021.0      1
topic1__ALL__UN__2011.0                              1
topic1__Christian democracy__parliament__2018.0      1
topic2__Christian democracy__parliament__2018.0      1
Name: count, Length: 272, dtype: int64


In [ ]:
# PREPARE TEXT FOR SUMMARIZATION
df_for_summarization = df.copy()

if "text_en" not in df_for_summarization.columns:
    df_for_summarization["text_en"] = None
if "text_clean" not in df_for_summarization.columns:
    df_for_summarization["text_clean"] = None


df_for_summarization["text_en_merged"] = ( #merge the column that contains the speech in parliament (text_en since it was translated), and text_clean for UN
    df_for_summarization["text_en"]
    .combine_first(df_for_summarization["text_clean"])
)
# some checks
df_for_summarization["text_en_merged"] = (
    df_for_summarization["text_en_merged"]
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df_for_summarization = df_for_summarization[
    df_for_summarization["text_en_merged"].notna()
    & (df_for_summarization["text_en_merged"].str.len() > 0)
    & (df_for_summarization["text_en_merged"] != "nan")
].copy()

print("Rows for summarization:", len(df_for_summarization))
print("\nRows by source:")
print(df_for_summarization["source"].value_counts(dropna=False))
print("\nRows by group_key:")
print(df_for_summarization["group_key"].value_counts(dropna=False))

assert "UN" in set(df_for_summarization["source"]), "UN is missing before summarization. Check cells 10-14."

Rows for summarization: 20019

Rows by source:
source
parliament    19799
UN              220
Name: count, dtype: int64

Rows by group_key:
group_key
topic2__M5S__parliament__2019.0                    100
topic2__Right-wing__parliament__2019.0             100
topic2__Social democracy__parliament__2019.0       100
topic2__Conservative__parliament__2020.0           100
topic2__Conservative__parliament__2018.0           100
                                                  ... 
topic1__ALL__UN__2006.0                              2
topic2__Christian democracy__parliament__2021.0      1
topic1__ALL__UN__2011.0                              1
topic1__Christian democracy__parliament__2018.0      1
topic2__Christian democracy__parliament__2018.0      1
Name: count, Length: 272, dtype: int64


In [ ]:
df_for_summarization.to_pickle('/content/df_for_summarization.pkl')

In [ ]:
df_for_summarization= pd.read_pickle('/content/df_for_summarization (1).pkl')

## summarization model

In [ ]:
# LOAD SUMMARIZATION MODEL
# The helper functions below avoid the worst truncation problem.
sum_model_name = "facebook/bart-large-cnn"
sum_tokenizer = AutoTokenizer.from_pretrained(sum_model_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(sum_model_name).to(device)
sum_model.eval()

print("Summarization model loaded:", sum_model_name)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Summarization model loaded: facebook/bart-large-cnn


In [ ]:
# SUMMARIZATION HELPER FUNCTIONS
def clean_text_for_summary(text):
    if pd.isna(text):
        return ""
    text = str(text).replace("\n", " ")
    text = " ".join(text.split())
    return text

def token_len(text):
    return len(sum_tokenizer.encode(str(text), add_special_tokens=False))

def chunk_text_by_tokens(text, max_tokens=900, overlap_tokens=80):
    """BART has a 1024-token input limit, so we keep chunks below that."""
    text = clean_text_for_summary(text)
    if not text:
        return []

    ids = sum_tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_tokens:
        return [text]

    chunks = []
    start = 0
    while start < len(ids):
        end = start + max_tokens
        chunk_ids = ids[start:end]
        chunks.append(sum_tokenizer.decode(chunk_ids, skip_special_tokens=True))
        if end >= len(ids):
            break
        start = end - overlap_tokens

    return chunks



In [ ]:
def summarize_texts_batched(texts, min_new_tokens=40, max_new_tokens=180, num_beams=2, batch_size=16):
    """Summarize a LIST of texts in GPU batches. Returns list of summaries."""
    #makes it much faster --> before 9 hrs of expected runtime
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = sum_tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=900,
            padding=True,
        ).to(device)
        with torch.no_grad():
            output = sum_model.generate(
                **inputs,
                min_new_tokens=min_new_tokens,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                length_penalty=1.0,
                early_stopping=True,
                no_repeat_ngram_size=3,
            )
        decoded = sum_tokenizer.batch_decode(output, skip_special_tokens=True)
        results.extend(decoded)
    return results

def summarize_one_speech_batched(text):
    """Same logic as summarize_one_speech but returns (chunks_list, needs_reduction)."""
    chunks = chunk_text_by_tokens(text, max_tokens=900, overlap_tokens=80)
    return chunks  # caller will batch these together across speeches


def summarize_all_speeches(texts):
    """
    Takes a list of raw speech texts.
    Returns a list of one summary per speech.
    Chunks long speeches, batches all chunk inference together.
    """
    # Step 1: chunk every speech
    all_chunks_per_speech = [chunk_text_by_tokens(t) for t in texts]

    # Step 2: flatten all chunks into one list, track which speech each belongs to
    flat_chunks = []
    speech_idx = []
    for i, chunks in enumerate(all_chunks_per_speech):
        for c in chunks:
            flat_chunks.append(c)
            speech_idx.append(i)

    # Step 3: batch summarize ALL chunks in one pass
    flat_summaries = summarize_texts_batched(
        flat_chunks, min_new_tokens=30, max_new_tokens=120, num_beams=2
    )

    # Step 4: group chunk summaries back per speech
    from collections import defaultdict
    grouped = defaultdict(list)
    for idx, summary in zip(speech_idx, flat_summaries):
        grouped[idx].append(summary)

    # Step 5: reduce multi-chunk speeches
    speech_summaries = []
    for i in range(len(texts)):
        chunks_summaries = grouped[i]
        if len(chunks_summaries) == 1:
            speech_summaries.append(chunks_summaries[0])
        else:
            speech_summaries.append(
                reduce_summaries_recursively(chunks_summaries, batch_size=5, final=False)
            )

    return speech_summaries


def reduce_summaries_recursively(summaries, batch_size=5, final=False):
    """Unchanged logic, but calls summarize_texts_batched instead of summarize_text."""
    summaries = [clean_text_for_summary(s) for s in summaries if clean_text_for_summary(s)]
    if not summaries:
        return ""

    joined = "\n".join(f"- {s}" for s in summaries)
    if token_len(joined) <= 900:
        return summarize_texts_batched(
            [joined],
            min_new_tokens=70 if final else 50,
            max_new_tokens=240 if final else 180,
        )[0]

    reduced = []
    for i in range(0, len(summaries), batch_size):
        batch_text = "\n".join(f"- {s}" for s in summaries[i:i + batch_size])
        reduced.append(batch_text)

    batch_results = summarize_texts_batched(
        reduced, min_new_tokens=50, max_new_tokens=170, num_beams=2
    )
    return reduce_summaries_recursively(batch_results, batch_size=batch_size, final=final)

In [ ]:
summaries = {}

groups = df_for_summarization.groupby("group_key", dropna=False)

for group_key, group_df in tqdm(groups, desc="Summarizing groups"):
    topic        = group_df["topic"].iloc[0]
    party_family = group_df["party_family"].iloc[0]
    source       = group_df["source"].iloc[0]
    date         = group_df["date_merged"].iloc[0]

    print(f"\nProcessing group: {group_key}")
    print(f"Source: {source} | Topic: {topic} | Party family: {party_family} | Speeches: {len(group_df)} | Date: {date}")

    texts = group_df["text_en_merged"].tolist()
    speech_summaries = summarize_all_speeches(texts)

    final_summary = reduce_summaries_recursively(speech_summaries, batch_size=5, final=True)

    summaries[group_key] = {
        "topic": topic,
        "party_family": party_family,
        "source": source,
        "date": date,
        "n_speeches": len(group_df),
        "individual_summaries": speech_summaries,
        "final_summary": final_summary,
    }

    print("Summary preview:", final_summary[:250], "...")

Summarizing groups:   0%|          | 0/272 [00:00<?, ?it/s]


Processing group: topic1__ALL__UN__2006.0
Source: UN | Topic: 1 | Party family: ALL | Speeches: 2 | Date: 2006.0
Summary preview: "There will be no peace until the palestinian question has been resolved," he says. "The situation in darfur is critical. We cannot stand by and watch" "We are for peace and solidarity. we are against the death penalty" he adds. 'We are against death ...

Processing group: topic1__ALL__UN__2011.0
Source: UN | Topic: 1 | Party family: ALL | Speeches: 1 | Date: 2011.0
Summary preview: Italy is the sixth top contributor to the united nations peacekeeping operations budget. It is also the top eu and western european and others group contributor of troops to the United Nations. It has a long tradition of mediation that has shaped our ...

Processing group: topic1__ALL__UN__2012.0
Source: UN | Topic: 1 | Party family: ALL | Speeches: 4 | Date: 2012.0
Summary preview: Italy is ready to accept the compulsory jurisdiction of the international court of justice. We fi

In [ ]:
# SAVE RESULTS AND CHECK THAT UN IS INCLUDED
results_df = pd.DataFrame([
    {
        "group": group_key,
        "topic": values["topic"],
        "party_family": values["party_family"],
        "source": values["source"],
        'date': values['date'],
        "n_speeches": values["n_speeches"],
        "final_summary": values["final_summary"]
    }
    for group_key, values in summaries.items()
])

results_df.to_csv('/content/summaries_output_FIXED.csv', index=False)
results_df.to_pickle('/content/summaries_output_FIXED.pkl')

print("Saved:")
print("/content/summaries_output_FIXED.csv")
print("/content/summaries_output_FIXED.pkl")
print("\nShape:", results_df.shape)
print("\nRows by source:")
print(results_df["source"].value_counts(dropna=False))
print("\nGroups:")
print(results_df[["group", "topic", "party_family", "source", 'date', "n_speeches"]].to_string(index=False))

assert "UN" in set(results_df["source"]), "UN is still missing from the final summaries."

Saved:
/content/summaries_output_FIXED.csv
/content/summaries_output_FIXED.pkl

Shape: (272, 7)

Rows by source:
source
parliament    243
UN             29
Name: count, dtype: int64

Groups:
                                          group  topic        party_family     source   date  n_speeches
                        topic1__ALL__UN__2006.0      1                 ALL         UN 2006.0           2
                        topic1__ALL__UN__2011.0      1                 ALL         UN 2011.0           1
                        topic1__ALL__UN__2012.0      1                 ALL         UN 2012.0           4
                        topic1__ALL__UN__2013.0      1                 ALL         UN 2013.0           7
                        topic1__ALL__UN__2014.0      1                 ALL         UN 2014.0           3
                        topic1__ALL__UN__2015.0      1                 ALL         UN 2015.0           7
                        topic1__ALL__UN__2016.0      1                 ALL

In [ ]:
results_df = pd.read_csv("/content/summaries_output_FIXED.csv")

In [ ]:
print(results_df.shape)
print(results_df[["source", "topic", "party_family", 'date', "n_speeches"]])

print("\nMissing final summaries:")
print(results_df["final_summary"].isna().sum())

print("\nSummary length:")
results_df["summary_words"] = results_df["final_summary"].astype(str).str.split().str.len()
print(results_df[["source", "topic", "party_family", 'date', "summary_words"]])

(272, 7)
         source  topic      party_family    date  n_speeches
0            UN      1               ALL  2006.0           2
1            UN      1               ALL  2011.0           1
2            UN      1               ALL  2012.0           4
3            UN      1               ALL  2013.0           7
4            UN      1               ALL  2014.0           3
..          ...    ...               ...     ...         ...
267  parliament      2  Social democracy  2018.0         100
268  parliament      2  Social democracy  2019.0         100
269  parliament      2  Social democracy  2020.0         100
270  parliament      2  Social democracy  2021.0         100
271  parliament      2  Social democracy  2022.0         100

[272 rows x 5 columns]

Missing final summaries:
0

Summary length:
         source  topic      party_family    date  summary_words
0            UN      1               ALL  2006.0             52
1            UN      1               ALL  2011.0             6

In [ ]:
import random

SAMPLE_N = 30
sample_df = results_df.sample(n=SAMPLE_N, random_state=42)

for i, (_, row) in enumerate(sample_df.iterrows(), 1):
    print(f"{'='*80}")
    print(f"[{i}/{SAMPLE_N}] Source: {row['source']} | "
          f"Topic: {row['topic']} | Party: {row['party_family']}")
    print(f"N speeches: {row['n_speeches']}")
    print("-"*80)
    print(row["final_summary"])
    word_count = len(row["final_summary"].split())
    print(f"\n  ↳ Words: {word_count}")
    print()

[1/30] Source: parliament | Topic: 1 | Party: Communist/Far-left
N speeches: 100
--------------------------------------------------------------------------------
North League wants to introduce a new law on immigration to make it easier to deport illegal immigrants. Italian PM: The motion is a sign of unprovincialization of Italian politics. 'I regret not to see in the Chamber an honourable member of the Russian Federation' European Parliament votes to makeIt easier for EU citizens to claim benefits from the EU.

  ↳ Words: 60

[2/30] Source: parliament | Topic: 1 | Party: Right-wing
N speeches: 100
--------------------------------------------------------------------------------
More than 2,000 migrants have landed on the Italian coast in a single week. The League has called for a suspension of the session of the European Council. Italian parliament is currently at its highest level since the end of the Second World War. Italian President: Helvetic confederation is an example of effici